# EXP-1 — Baseline Vector Retrieval

The simplest possible RAG setup. Embed the question, fetch top-k chunks by cosine similarity, pass them to the LLM. This is the control experiment that every other run is compared against.

## Setup

In [ ]:
import os
import sys
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.document_loaders import TextLoader

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings('ignore')

print("all imports loaded")


all imports loaded


In [ ]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [7]:

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [21]:
import dagshub
dagshub.init(repo_owner='DataShoaib', repo_name='hr-policy-chatbot-rag', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ec7e6489-faf7-4e11-a691-eece3d34213a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=992de78a972892a5f8fe46c1c9c0f6c6d90d5bb150fc727658440d833a1d0236




Accessing as DataShoaib

Initialized MLflow to track repo "DataShoaib/hr-policy-chatbot-rag"

Repository DataShoaib/hr-policy-chatbot-rag initialized!

## Load and Prepare Data

In [ ]:
# adjust the path and glob pattern to match the folder structure
loader = DirectoryLoader("../data/policies",glob="**/*.md",loader_cls=TextLoader )
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [30]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [31]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3278.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [32]:

print(f"eval set: {len(dataset['question'])} question-answer pairs")


eval set: 10 question-answer pairs


In [39]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  
    temperature=0
)

In [34]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [52]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [53]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            docs = get_docs_fn(question)
            answers.append(answer)
            contexts.append([d.page_content for d in docs])
        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    })

    # max_workers=1 means sequential — no parallel calls, no rate limit hits
    # timeout=120 because Groq free tier can be slow sometimes
    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,   # if one question fails,we still want to get metrics on the rest of the eval ser
    )
    return result

In [62]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
            # numeric_only=True — sirf score columns average karo, string columns ignore karo
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment

In [55]:
# plain cosine similarity search, top 3 chunks, no extra tricks
retriever = db.as_retriever(search_kwargs={"k": 3})

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# sanity check before running the full eval
print(chain.invoke("What is the leave policy?"))


The leave policy includes the following details:

- Earned Leave / Privilege Leave (PL/EL): 18 days per calendar year, with accrual of 1.5 days per month after completing 6 months of service.
- Work From Home (WFH) Leave: Up to 3 days per week for eligible roles, with manager approval required.
- Leave Without Pay (LWP): Applicable when all leave balance is exhausted, requires explicit approval from Department Head and HR, and affects annual increment calculation (pro-rated).
- Leave encashment rules: Allowed during service (maximum 15 days per year) and at full & final settlement.


In [ ]:
result = evaluate_rag(chain, retriever.invoke, dataset)
latency = measure_latency(chain)

Evaluating: 100%|██████████| 30/30 [10:28<00:00, 20.95s/it]


In [60]:
result,latency

({'faithfulness': 0.9750, 'context_precision': 0.4000, 'context_recall': 0.3500},
 0.664)

In [63]:
log_to_mlflow(
    run_name="EXP-1-BASELINE",
    result=result,
    latency=latency,
    retriever_type="vector-similarity",
    top_k=3,
)

🏃 View run EXP-1-BASELINE at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/0/runs/4180a2f39b064ca8b9b9c889f05fda20
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/0
